# 05. Rule-Based Prompt Evaluation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/05_rule_based_prompt_eval.ipynb)

This notebook introduces deterministic prompt evaluation for structured tasks. The goal is to build cheap, transparent checks that make prompt iteration safer before you reach for heavier judge models.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Learning goals

By the end you will be able to:

- create a tiny prompt benchmark with in-notebook data
- compare exact match, regex, and schema-aware scorers
- turn a benchmark into a lightweight regression suite
- explain both the usefulness and the limits of deterministic grading

## Why this matters

Prompt work often becomes anecdotal: a change looks better on one example, so it ships. Deterministic graders help when outputs are structured enough to check mechanically. They are not universal, but they are fast, cheap, and auditable.

In [ ]:
import json
import random
import re
import statistics
from copy import deepcopy
from IPython.display import Markdown, display

random.seed(7)

def show_table(rows, columns=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return
    if columns is None:
        columns = list(rows[0].keys())
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    display(Markdown("\n".join([header, divider] + body)))

def mean(values):
    values = list(values)
    return round(sum(values) / len(values), 3) if values else 0.0

## Step 1: Build a tiny benchmark dataset

We will evaluate prompt variants on support-ticket triage. Each task has a raw ticket and a gold structured answer. This is exactly the kind of problem where deterministic checks are often enough.

In [ ]:
benchmark_cases = [
    {
        "id": "t1",
        "ticket": "I was charged twice for the annual plan and need a refund today.",
        "gold": {
            "department": "billing",
            "priority": "high",
            "needs_response": True,
            "issue_type": "duplicate_charge",
        },
    },
    {
        "id": "t2",
        "ticket": "My package says delivered but nothing arrived. Please investigate.",
        "gold": {
            "department": "shipping",
            "priority": "high",
            "needs_response": True,
            "issue_type": "missing_delivery",
        },
    },
    {
        "id": "t3",
        "ticket": "The dashboard export button does nothing in Chrome after the latest update.",
        "gold": {
            "department": "technical",
            "priority": "medium",
            "needs_response": True,
            "issue_type": "ui_bug",
        },
    },
    {
        "id": "t4",
        "ticket": "Can someone explain the difference between the starter and team plans?",
        "gold": {
            "department": "sales",
            "priority": "low",
            "needs_response": True,
            "issue_type": "plan_question",
        },
    },
    {
        "id": "t5",
        "ticket": "I updated my card details successfully. No action needed now, thanks.",
        "gold": {
            "department": "billing",
            "priority": "low",
            "needs_response": False,
            "issue_type": "account_update",
        },
    },
    {
        "id": "t6",
        "ticket": "The reset-password link keeps looping back to the login page.",
        "gold": {
            "department": "technical",
            "priority": "high",
            "needs_response": True,
            "issue_type": "auth_bug",
        },
    },
    {
        "id": "t7",
        "ticket": "We need an invoice with our tax ID before procurement can pay.",
        "gold": {
            "department": "billing",
            "priority": "medium",
            "needs_response": True,
            "issue_type": "invoice_request",
        },
    },
    {
        "id": "t8",
        "ticket": "Please close my duplicate workspace. I already migrated to the new one.",
        "gold": {
            "department": "technical",
            "priority": "medium",
            "needs_response": True,
            "issue_type": "workspace_cleanup",
        },
    },
]

show_table(
    [
        {
            "id": case["id"],
            "ticket": case["ticket"][:55] + "...",
            "gold": json.dumps(case["gold"], sort_keys=True),
        }
        for case in benchmark_cases
    ],
    columns=["id", "ticket", "gold"],
)

## Step 2: Define prompt variants

To keep the notebook lightweight and reproducible, the functions below simulate outputs from different prompt templates. In a real workflow you would swap these functions for model calls, but the evaluation logic would stay the same.

In [ ]:
SCHEMA_KEYS = ["department", "priority", "needs_response", "issue_type"]

def canonical_response(case):
    return json.dumps(case["gold"], sort_keys=True)

def prompt_loose(case):
    result = deepcopy(case["gold"])
    if case["id"] in {"t2", "t6"}:
        result["needs_response"] = "yes" if result["needs_response"] else "no"
    text = json.dumps(result, indent=2)
    return "I triaged the ticket like this:\n" + text + "\nCustomer-facing follow-up is recommended."

def prompt_json_only(case):
    result = deepcopy(case["gold"])
    reordered = {
        "priority": result["priority"],
        "department": result["department"],
        "issue_type": result["issue_type"],
        "needs_response": result["needs_response"],
    }
    return json.dumps(reordered, indent=2)

def prompt_schema_first(case):
    result = deepcopy(case["gold"])
    if case["id"] == "t5":
        result["issue_type"] = "account"
    return json.dumps(result, sort_keys=True)

TEMPLATES = {
    "loose": prompt_loose,
    "json_only": prompt_json_only,
    "schema_first": prompt_schema_first,
}

In [ ]:
preview_case = benchmark_cases[1]
for name, fn in TEMPLATES.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print(fn(preview_case))

## Step 3: Exact match scoring

Exact match is the cheapest possible evaluator. It is useful when format and content both matter. It is also brittle: valid answers can fail because of key order or harmless wording changes.

In [ ]:
def exact_match_score(case, response):
    target = canonical_response(case)
    return 1.0 if response.strip() == target else 0.0

exact_rows = []
for case in benchmark_cases[:3]:
    exact_rows.append(
        {
            "id": case["id"],
            "loose": exact_match_score(case, prompt_loose(case)),
            "json_only": exact_match_score(case, prompt_json_only(case)),
            "schema_first": exact_match_score(case, prompt_schema_first(case)),
        }
    )

show_table(exact_rows)

## Step 4: Regex extraction

Regex grading is a middle ground. It can recover fields from noisy text, but each field needs custom parsing logic and the grader becomes harder to maintain as outputs diversify.

In [ ]:
FIELD_PATTERNS = {
    "department": r'"department"\s*:\s*"([^"]+)"',
    "priority": r'"priority"\s*:\s*"([^"]+)"',
    "needs_response": r'"needs_response"\s*:\s*(true|false|"yes"|"no")',
    "issue_type": r'"issue_type"\s*:\s*"([^"]+)"',
}

def normalize_regex_value(field, value):
    if field == "needs_response":
        value = value.strip('"').lower()
        return value in {"true", "yes"}
    return value.lower()

def regex_extract(text):
    extracted = {}
    for field, pattern in FIELD_PATTERNS.items():
        match = re.search(pattern, text, re.I)
        if match:
            extracted[field] = normalize_regex_value(field, match.group(1))
    return extracted

def regex_score(case, response):
    extracted = regex_extract(response)
    gold = case["gold"]
    matches = 0
    for field in SCHEMA_KEYS:
        if field not in extracted:
            continue
        gold_value = gold[field] if field == "needs_response" else str(gold[field]).lower()
        if extracted[field] == gold_value:
            matches += 1
    return matches / len(SCHEMA_KEYS)

regex_rows = []
for case in benchmark_cases[:3]:
    regex_rows.append(
        {
            "id": case["id"],
            "loose": round(regex_score(case, prompt_loose(case)), 2),
            "json_only": round(regex_score(case, prompt_json_only(case)), 2),
            "schema_first": round(regex_score(case, prompt_schema_first(case)), 2),
        }
    )

show_table(regex_rows)

## Step 5: Schema-aware scoring

Schema-aware scoring tries to parse the structured payload, validate required keys and types, and then compare field values. This is often the best deterministic option when you expect JSON-like outputs.

In [ ]:
def extract_json_object(text):
    match = re.search(r"\{.*\}", text, re.S)
    return match.group(0) if match else None

def schema_score(case, response):
    payload = extract_json_object(response)
    if payload is None:
        return {
            "valid_json": False,
            "valid_schema": False,
            "field_accuracy": 0.0,
        }
    try:
        data = json.loads(payload)
    except json.JSONDecodeError:
        return {
            "valid_json": False,
            "valid_schema": False,
            "field_accuracy": 0.0,
        }

    required_keys_present = all(key in data for key in SCHEMA_KEYS)
    type_checks = [
        isinstance(data.get("department"), str),
        isinstance(data.get("priority"), str),
        isinstance(data.get("needs_response"), bool),
        isinstance(data.get("issue_type"), str),
    ]
    valid_schema = required_keys_present and all(type_checks)

    field_hits = 0
    for key in SCHEMA_KEYS:
        if key in data and data[key] == case["gold"][key]:
            field_hits += 1

    return {
        "valid_json": True,
        "valid_schema": valid_schema,
        "field_accuracy": field_hits / len(SCHEMA_KEYS),
    }

schema_rows = []
for case in benchmark_cases[:3]:
    row = {"id": case["id"]}
    for name, fn in TEMPLATES.items():
        result = schema_score(case, fn(case))
        row[name] = f"{result['field_accuracy']:.2f} | schema={result['valid_schema']}"
    schema_rows.append(row)

show_table(schema_rows)

## Step 6: Run the benchmark

A benchmark should summarize the trade-offs clearly. Here we compare task performance and a crude cost proxy based on output length.

In [ ]:
def evaluate_template(name, fn):
    per_case = []
    for case in benchmark_cases:
        response = fn(case)
        schema_result = schema_score(case, response)
        per_case.append(
            {
                "id": case["id"],
                "exact": exact_match_score(case, response),
                "regex": regex_score(case, response),
                "schema": schema_result["field_accuracy"],
                "valid_schema": schema_result["valid_schema"],
                "chars": len(response),
            }
        )
    return {
        "template": name,
        "exact_mean": mean(row["exact"] for row in per_case),
        "regex_mean": mean(row["regex"] for row in per_case),
        "schema_mean": mean(row["schema"] for row in per_case),
        "schema_pass_rate": mean(1.0 if row["valid_schema"] else 0.0 for row in per_case),
        "mean_chars": round(statistics.mean(row["chars"] for row in per_case), 1),
    }, per_case

benchmark_summary = []
benchmark_details = {}

for name, fn in TEMPLATES.items():
    summary_row, per_case_rows = evaluate_template(name, fn)
    benchmark_summary.append(summary_row)
    benchmark_details[name] = per_case_rows

show_table(
    benchmark_summary,
    columns=["template", "exact_mean", "regex_mean", "schema_mean", "schema_pass_rate", "mean_chars"],
)

## Step 7: Inspect scorer disagreements

This is where deterministic evaluation becomes educational. The json_only prompt is semantically correct, but exact match still punishes it because the keys appear in a different order.

In [ ]:
disagreement_rows = []
for case in benchmark_cases:
    response = prompt_json_only(case)
    schema_result = schema_score(case, response)
    disagreement_rows.append(
        {
            "id": case["id"],
            "exact": exact_match_score(case, response),
            "regex": regex_score(case, response),
            "schema": schema_result["field_accuracy"],
            "valid_schema": schema_result["valid_schema"],
        }
    )

show_table(disagreement_rows)

## Step 8: Turn the benchmark into a regression suite

Regression checks should be intentionally simple. They are guardrails, not a complete theory of quality.

In [ ]:
critical_case_ids = {"t1", "t4", "t7"}

def run_regression_suite(name, fn, required_schema=0.95):
    rows = []
    for case in benchmark_cases:
        response = fn(case)
        schema_result = schema_score(case, response)
        rows.append(
            {
                "id": case["id"],
                "schema": schema_result["field_accuracy"],
                "valid_schema": schema_result["valid_schema"],
            }
        )

    schema_mean = mean(row["schema"] for row in rows)
    critical_pass = all(
        row["valid_schema"] and row["schema"] == 1.0
        for row in rows
        if row["id"] in critical_case_ids
    )
    return {
        "template": name,
        "schema_mean": schema_mean,
        "critical_cases_pass": critical_pass,
        "release_ready": schema_mean >= required_schema and critical_pass,
    }

suite_rows = [run_regression_suite(name, fn) for name, fn in TEMPLATES.items()]
show_table(suite_rows)

## Step 9: Simulate a bad prompt edit

If a prompt edit silently drops a required field, a cheap regression suite should fail immediately.

In [ ]:
def prompt_schema_first_broken(case):
    result = deepcopy(case["gold"])
    if case["id"] in {"t3", "t7"}:
        result.pop("needs_response")
    return json.dumps(result, sort_keys=True)

show_table([run_regression_suite("schema_first_broken", prompt_schema_first_broken)])

In [ ]:
for case in benchmark_cases:
    if case["id"] in {"t3", "t7"}:
        print(case["id"])
        print(prompt_schema_first_broken(case))
        print()

## What deterministic graders are good at

Use them when you need:

- cheap checks on every prompt edit
- stable scoring for structured outputs
- transparent pass or fail logic
- fast feedback before running expensive judge-model experiments

## Where deterministic graders break down

They struggle when:

- multiple phrasings are equally good
- answers are open-ended or stylistic
- the grader becomes more complex than the task
- semantic correctness matters more than literal format

## Suggested extensions

Try three follow-ups:

1. replace the stand-in prompt functions with real model calls
2. add a new field such as customer sentiment and update each grader
3. log scorer disagreements so future prompt edits are easier to debug